In [3]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
from scipy.io import mmwrite

In [1]:
def get_raw_counts(adata):
    if adata.raw is None:
        raise ValueError("adata.raw is None. 你需要把 raw counts 保存进 raw 才能做 edgeR。")
    X = adata.raw.X
    if not sp.issparse(X):
        X = sp.csr_matrix(X)
    return X.tocsr()

def pseudobulk_by_donor_raw(adata, donor_col="donor_id", meta_cols=("disease","Primary.Genetic.Diagnosis","sex","Region_x","Sample"),
                           min_cells=30):
    X = get_raw_counts(adata)
    obs = adata.obs.copy()
    # donor -> cell indices
    donor_groups = obs.groupby(donor_col).indices

    kept_donors, mats, n_cells = [], [], []
    for d, idx in donor_groups.items():
        if len(idx) < min_cells:
            continue
        kept_donors.append(d)
        mats.append(X[idx, :].sum(axis=0))
        n_cells.append(len(idx))

    if len(mats) == 0:
        top = obs[donor_col].value_counts().head(10)
        raise ValueError(f"No donors passed min_cells={min_cells}. Top donor cell counts:\n{top}")

    PB = sp.vstack([sp.csr_matrix(m) for m in mats]).tocsr()  # donors x genes

    meta = (obs.loc[obs[donor_col].isin(kept_donors)]
              .groupby(donor_col)
              .agg({c: "first" for c in meta_cols if c in obs.columns}))
    meta = meta.loc[kept_donors].copy()
    meta["donor_id"] = kept_donors
    meta["n_cells"] = n_cells
    meta.index = pd.Index([str(d) for d in kept_donors], name="sample_id")
    return PB, meta

def export_for_edger(PB_donor_by_gene, meta, gene_names, outdir, prefix):
    os.makedirs(outdir, exist_ok=True)
    # edgeR 常用 genes x samples
    PB = PB_donor_by_gene.T.tocoo()

    mmwrite(os.path.join(outdir, f"{prefix}.counts.mtx"), PB)
    pd.Series(gene_names, name="gene").to_csv(os.path.join(outdir, f"{prefix}.genes.tsv"),
                                              sep="\t", index=False, header=False)
    pd.Series(meta.index.astype(str), name="sample").to_csv(os.path.join(outdir, f"{prefix}.samples.tsv"),
                                                           sep="\t", index=False, header=False)
    meta.to_csv(os.path.join(outdir, f"{prefix}.meta.csv"), index=True)

In [4]:
#Load data
adata = sc.read_h5ad("local.h5ad")

In [8]:
ct = adata.obs["cell_type"].astype(str)

mask_cm = ct.eq("cardiac muscle cell")
mask_ec = ct.eq("endothelial cell")

# ===== CM =====
ad_cm = adata[mask_cm].copy()
print("CM donor cell counts top:\n", ad_cm.obs["donor_id"].value_counts().head(10))

PB_cm, meta_cm = pseudobulk_by_donor_raw(ad_cm, min_cells=100)
export_for_edger(PB_cm, meta_cm, ad_cm.raw.var_names, outdir="edger_export", prefix="CM")

# ===== EC =====
ad_ec = adata[mask_ec].copy()
print("EC donor cell counts top:\n", ad_ec.obs["donor_id"].value_counts().head(10))

PB_ec, meta_ec = pseudobulk_by_donor_raw(ad_ec, min_cells=100)
export_for_edger(PB_ec, meta_ec, ad_ec.raw.var_names, outdir="edger_export", prefix="EC")


CM donor cell counts top:
 donor_id
H5     20568
H6     13564
H79    12609
H7     11206
H3     10824
H4     10130
D5      9100
H12     8993
H49     8531
D4      8482
Name: count, dtype: int64


/var/tmp/pbs.1552514.pbs-7/ipykernel_1045989/1241551957.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  donor_groups = obs.groupby(donor_col).indices
/var/tmp/pbs.1552514.pbs-7/ipykernel_1045989/1241551957.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(donor_col)


EC donor cell counts top:
 donor_id
H06    4405
H03    4150
H33    3852
H02    3114
H19    2968
H01    2921
H07    2790
DP1    2700
H28    2692
H25    2684
Name: count, dtype: int64


/var/tmp/pbs.1552514.pbs-7/ipykernel_1045989/1241551957.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  donor_groups = obs.groupby(donor_col).indices
/var/tmp/pbs.1552514.pbs-7/ipykernel_1045989/1241551957.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(donor_col)
